# 🔥 Module 8: CNN-Based Land Cover Classification

Welcome to **Module 8** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **CNN Fundamentals**: Understand 2D convolution, feature maps, batch normalization, and pooling.
2. **Multi-Band Input**: Feed 5-channel multispectral satellite patches (Blue, Green, Red, NIR, SWIR) into deep convolutional layers.
3. **Train SatelliteLandCoverCNN**: Classify patches into 5 categories: **Forest**, **Bare Land / Deforested**, **Agricultural Land**, **Urban Area**, and **Water**.
4. **Inspect Feature Maps**: Visualize layer activations (low-level edges, mid-level textures, high-level semantics).
5. **Transfer Learning**: Explore ResNet architectures adapted for remote sensing.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_08_cnn_classification.patch_dataset import (
    LandCoverPatchDataset,
    create_dataloaders
)
from modules.module_08_cnn_classification.cnn_models import (
    SatelliteLandCoverCNN,
    SatelliteResNet18
)
from modules.module_08_cnn_classification.trainer import (
    train_land_cover_model,
    evaluate_model
)

%matplotlib inline
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Deep Learning environment ready on device: [{device}]")

## 1. Create 5-Class Multi-Band Patch DataLoaders

We partition patches into train (70%), validation (15%), and test (15%) splits across the 5 land-cover classes.

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    total_samples=1500,
    batch_size=32,
    patch_size=32,
    seed=42
)
class_names = LandCoverPatchDataset.CLASS_NAMES
print("Classes:", class_names)
print(f"Train samples: {len(train_loader.dataset)} | Val samples: {len(val_loader.dataset)} | Test samples: {len(test_loader.dataset)}")

## 2. Visualize Sample Patches Across All 5 Classes

Inspect the distinct visual and spectral differences among the classes.

In [ ]:
def make_rgb(p):
    def s(arr):
        p2, p98 = np.percentile(arr, (2, 98))
        return np.clip((arr - p2) / (p98 - p2 + 1e-8), 0.0, 1.0)
    return np.stack([s(p[2]), s(p[1]), s(p[0])], axis=-1)

batch_x, batch_y = next(iter(train_loader))
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), constrained_layout=True)

for i in range(5):
    axes[i].imshow(make_rgb(batch_x[i].numpy()))
    axes[i].set_title(class_names[batch_y[i].item()], fontweight="bold")
    axes[i].axis("off")

plt.show()

## 3. Train SatelliteLandCoverCNN

We train the 3-stage CNN using CrossEntropyLoss and Adam optimizer.

In [ ]:
model = SatelliteLandCoverCNN(in_channels=5, num_classes=5)
trained_model, history = train_land_cover_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-3,
    device=device
)

## 4. Plot Loss and Accuracy Convergence Curves

Verify steady gradient descent convergence and absence of overfitting.

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
ax1.plot(epochs_range, history['val_loss'], 'r--s', label='Val Loss')
ax1.set_title('Cross-Entropy Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend()

ax2.plot(epochs_range, history['train_acc'], 'g-o', label='Train Acc')
ax2.plot(epochs_range, history['val_acc'], 'm--s', label='Val Acc')
ax2.set_title('Accuracy (%)', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend()

plt.show()

## 5. Evaluate Confusion Matrix on Unseen Test Patches

Verify classification performance across all 5 land-cover categories.

In [ ]:
eval_results = evaluate_model(trained_model, test_loader, class_names, device=device)
cm = eval_results['confusion_matrix']
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(7, 6))
plt.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.xticks(range(len(class_names)), class_names, rotation=35, ha='right', fontweight='bold')
plt.yticks(range(len(class_names)), class_names, fontweight='bold')
plt.title(f"Confusion Matrix (Overall Test Acc: {eval_results['accuracy']*100:.1f}%)", fontweight='bold')
plt.colorbar()
plt.show()

print(f"Inference latency: {eval_results['latency_per_patch_ms']:.3f} ms / patch")

## 6. Inspect Convolutional Feature Maps

See what the CNN has learned:
- **Stage 1 (16x16)**: Low-level edges and color contrast.
- **Stage 2 (8x8)**: Texture patterns and boundaries.
- **Stage 3 (4x4)**: High-level semantic land cover representations.

In [ ]:
sample_x = batch_x[0:1].to(device)
trained_model.eval()
with torch.no_grad():
    _, feat_maps = trained_model.forward_with_features(sample_x)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
axes[0].imshow(make_rgb(sample_x[0].cpu().numpy()))
axes[0].set_title(f"Input: {class_names[batch_y[0].item()]}", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(feat_maps['Stage 1 (Edges & Color)'][0].mean(dim=0).cpu().numpy(), cmap='viridis')
axes[1].set_title('Stage 1 (16x16)', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(feat_maps['Stage 2 (Textures & Boundaries)'][0].mean(dim=0).cpu().numpy(), cmap='viridis')
axes[2].set_title('Stage 2 (8x8)', fontweight='bold')
axes[2].axis('off')

axes[3].imshow(feat_maps['Stage 3 (Semantic Land Cover)'][0].mean(dim=0).cpu().numpy(), cmap='magma')
axes[3].set_title('Stage 3 (4x4)', fontweight='bold')
axes[3].axis('off')

plt.show()